# 任务委派模式
监督者模式的优化实现,裁剪了agent之间通信时不必要的消息

## 魔法
handoff_tool工具的入参增加了task_description.这个参数是supervisor图中模型总结出的下一个子图的任务描述.
返回值Command中,使用Send跳转到指定节点,以替代原先直接goto的方式(会传递全部state的快照)

In [2]:


from typing import Annotated

from langgraph.graph import MessagesState
from langgraph.prebuilt import InjectedState
from langgraph.types import Command, Send
from langchain_core.tools import tool

def create_task_description_handoff_tool(
    *, agent_name: str, description: str | None = None
):
    """创建带任务描述的交接工具"""
    name = f"transfer_to_{agent_name}"
    description = description or f"Ask {agent_name} for help."

    @tool(name, description=description)
    def handoff_tool(
        # 由 Supervisor LLM 填充的任务描述
        task_description: Annotated[
            str,
            "Description of what the next agent should do, including all of the relevant context.",
        ],
        state: Annotated[MessagesState, InjectedState],
    ) -> Command:
        task_description_message = {"role": "user", "content": task_description}
        agent_input = {**state, "messages": [task_description_message]}
        return Command(
            # 使用 Send 将特定输入发送给目标 Agent
            goto=[Send(agent_name, agent_input)],
            graph=Command.PARENT,
        )

    return handoff_tool

# 创建带任务描述的交接工具
assign_to_research_agent_with_description = create_task_description_handoff_tool(
    agent_name="research_agent",
    description="Assign task to a researcher agent.",
)

assign_to_math_agent_with_description = create_task_description_handoff_tool(
    agent_name="math_agent",
    description="Assign task to a math agent.",
)

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="glm-4.7",
    api_key="d1de7475090740ebb6c887167a6a8b98.2WUSfYbmgQwYLKF4",
    base_url="https://open.bigmodel.cn/api/paas/v4/",  # 智谱OpenAI兼容地址，末尾斜杠不能丢
)

# 创建research agent
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# 创建搜索工具
web_search = TavilySearch(
    max_results=3,
    tavily_api_key="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",
)

# 测试搜索工具
web_search_results = web_search.invoke("who is the mayor of NYC?")
print(web_search_results["results"][0]["content"])

research_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "You are a research agent.\n\n"
        "INSTRUCTIONS:\n"
        "- Assist ONLY with research-related tasks, DO NOT do any math\n"
        "- After you're done with your tasks, respond to the supervisor directly\n"
        "- Respond ONLY with the results of your work, do NOT include ANY other text."
    ),
    name="research_agent",
)


# 创建mathagent
def add(a: float, b: float):
    """Add two numbers."""
    return a + b

def multiply(a: float, b: float):
    """Multiply two numbers."""
    return a * b

def divide(a: float, b: float):
    """Divide two numbers."""
    return a / b

math_agent = create_agent(
    model=llm,
    tools=[add, multiply, divide],
    system_prompt=(
        "You are a math agent.\n\n"
        "INSTRUCTIONS:\n"
        "- Assist ONLY with math-related tasks\n"
        "- After you're done with your tasks, respond to the supervisor directly\n"
        "- Respond ONLY with the results of your work, do NOT include ANY other text."
    ),
    name="math_agent",
)


from langchain.agents import create_agent

supervisor_agent = create_agent(
    model=llm,
    tools=[assign_to_research_agent_with_description, assign_to_math_agent_with_description],
    system_prompt=(
        "You are a supervisor managing two agents:\n"
        "- a research agent. Assign research-related tasks to this agent\n"
        "- a math agent. Assign math-related tasks to this agent\n"
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Do not do any work yourself."
        "涉及到数学计算,必须调用match agent"
    ),
    name="supervisor",
)


from langgraph.graph import END, START, StateGraph

# 定义多智能体 Supervisor 图
supervisor = (
    StateGraph(MessagesState)
    # 添加节点
    .add_node(supervisor_agent, destinations=("research_agent", "math_agent", END))
    .add_node(research_agent)
    .add_node(math_agent)
    # 添加边
    .add_edge(START, "supervisor")
    # Worker Agent 执行完毕后返回 Supervisor
    .add_edge("research_agent", "supervisor")
    .add_edge("math_agent", "supervisor")
    .compile()
)

The current mayor is Zohran Mamdani, who was elected on November 4, 2025, and took office shortly after midnight on January 1, 2026.

## History

[edit]

See also: List of mayors of New York City [...] of the mayor. Cornelius W. Lawrence, a Democrat, was elected that year. [...] | Seal of the City of New York |
| Flag of the mayor of New York City |
| Incumbent Zohran Mamdani since January 1, 2026 |
| Government of New York City |
| Style "Style (form of address)") | His Honor "Honour (style)"); Mr. Mayor (informal) |
| Residence | Gracie Mansion |
| Seat | New York City Hall |
| Term length | Four years, renewable once consecutively |
| Constituting instrument | New York City Charter |
| Inaugural holder | Thomas Willett |


In [4]:
res = supervisor.invoke({
        "messages": [
            {
                "role": "user",
                "content": "find US and New York state GDP in 2024. what % of US GDP was New York state?",
            }
        ]
    })


In [ ]:
for msg in res["messages"]:
    msg.pretty_print() 


content='find US and New York state GDP in 2024. what % of US GDP was New York state?' additional_kwargs={} response_metadata={} id='6068cc0f-ccfb-4006-82f7-616ee1a49c73'
content='Find the GDP (Gross Domestic Product) figures for both the United States and New York state for the year 2024. Please provide:\n1. US total GDP for 2024 (in current dollars)\n2. New York state GDP for 2024 (in current dollars)\n3. Any relevant sources or official data sources used\n\nThis data will be used to calculate what percentage of US GDP was contributed by New York state in 2024.' additional_kwargs={} response_metadata={} id='d195241c-0717-4fe7-b19a-89af50cf9a3b'
content="I'll help you find the GDP figures for both the United States and New York state for 2024. Let me search for this information." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 234, 'prompt_tokens': 1791, 'total_tokens': 2025, 'completion_tokens_details': {'accepted_prediction_tokens': None, 

In [13]:
for chunk in supervisor.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "find US and New York state GDP in 2024. what % of US GDP was New York state?",
            }
        ]
    },
    subgraphs=True
):
    print(chunk)

(('supervisor:ef786995-c13f-79aa-4d44-b8fb3ad9a31c',), {'model': {'messages': [AIMessage(content="I'll assign this research task to find the GDP data you need.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 340, 'total_tokens': 524, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 83, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256}}, 'model_provider': 'openai', 'model_name': 'glm-4.7', 'system_fingerprint': None, 'id': '20260826160730b6a7eb0ac14b4fd1', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--01a03d1c-1aca-75f2-ae4c-da80f658730f-0', tool_calls=[{'name': 'transfer_to_research_agent', 'args': {'task_description': "Find the following economic data for 2024:\n1. US GDP (Gross Domestic Product) for the year 2024\n2. New York state GDP for 